# create table

In [0]:
%sql
CREATE OR REPLACE TABLE `medallion_architecture`.`gold`.`gold_facility` (
  `facility_id` STRING COMMENT 'Stable internal golden facility ID (using unique_id)',
  `canonical_name` STRING COMMENT 'Cleaned and standardized facility name',
  `alternate_names` ARRAY<STRING> COMMENT 'Other names found across sources',
  `organization_type` STRING COMMENT 'Hospital, clinic, diagnostic center, medical college, chain, etc.',
  `facility_type` STRING COMMENT 'Normalized facility type: hospital, clinic, dentist, pharmacy, etc.',
  `operator_type` STRING COMMENT 'Public, private, charitable, trust, government, corporate, NGO, academic, etc.',
  `is_public` BOOLEAN COMMENT 'Boolean flag for public/government facility',
  `is_private` BOOLEAN COMMENT 'Boolean flag for private facility',
  `is_academic` BOOLEAN COMMENT 'Boolean flag for teaching hospital or medical college affiliation',
  `description_summary` STRING COMMENT 'Short cleaned facility description',
  `status` STRING COMMENT 'Active, inactive, closed, unknown, duplicate, needs review',
  `gold_confidence_score` DOUBLE COMMENT 'Overall confidence in the golden identity record (0-1)',
  `last_verified_at` TIMESTAMP COMMENT 'Most recent date this facility identity was verified',
  `data_quality_flags` ARRAY<STRING> COMMENT 'Array of issues: duplicate candidate, missing address, conflicting type, weak source',
  `created_at` TIMESTAMP COMMENT 'Record creation timestamp',
  `updated_at` TIMESTAMP COMMENT 'Record last update timestamp'
)
COMMENT 'Gold layer master table for unique healthcare facilities. One row per canonical facility.';

# Insert Data

In [0]:
%sql
INSERT INTO `medallion_architecture`.`gold`.`gold_facility`
SELECT
  `unique_id` AS `facility_id`,
  TRIM(`name`) AS `canonical_name`,
  CAST(NULL AS ARRAY<STRING>) AS `alternate_names`,
  CASE 
    WHEN LOWER(`organization_type`) = 'facility' THEN 'Healthcare Facility'
    WHEN `organization_type` IS NULL THEN 'Unknown'
    ELSE TRIM(`organization_type`)
  END AS `organization_type`,
  CASE 
    WHEN LOWER(`facility_type_id`) = 'hospital' THEN 'Hospital'
    WHEN LOWER(`facility_type_id`) = 'clinic' THEN 'Clinic'
    WHEN LOWER(`facility_type_id`) = 'dentist' THEN 'Dental Clinic'
    WHEN LOWER(`facility_type_id`) IN ('pharmacy', 'farmacy') THEN 'Pharmacy'
    WHEN LOWER(`facility_type_id`) = 'doctor' THEN 'Doctor Office'
    WHEN LOWER(`facility_type_id`) = 'nursing_home' THEN 'Nursing Home'
    WHEN `facility_type_id` IS NULL THEN 'Unknown'
    WHEN `facility_type_id` LIKE '%coordinates%' 
      OR `facility_type_id` LIKE '%http%' 
      OR `facility_type_id` LIKE '08f%'
      OR `facility_type_id` RLIKE '^[0-9.]+$'
      OR `facility_type_id` = 'kie' THEN 'Unknown'
    ELSE TRIM(`facility_type_id`)
  END AS `facility_type`,
  CASE 
    WHEN LOWER(`operator_type_id`) = 'private' THEN 'Private'
    WHEN LOWER(`operator_type_id`) IN ('public', 'government') THEN 'Public/Government'
    WHEN `operator_type_id` IS NULL THEN 'Unknown'
    WHEN `operator_type_id` LIKE '%http%' 
      OR `operator_type_id` LIKE '08f%'
      OR `operator_type_id` RLIKE '^[0-9.]+$'
      OR `operator_type_id` = 'true'
      OR `operator_type_id` = 'kie'
      OR `operator_type_id` LIKE '%coordinates%' THEN 'Unknown'
    ELSE TRIM(`operator_type_id`)
  END AS `operator_type`,
  CASE 
    WHEN LOWER(`operator_type_id`) IN ('public', 'government') THEN TRUE
    WHEN `affiliation_type_ids` LIKE '%government%' THEN TRUE
    ELSE FALSE
  END AS `is_public`,
  CASE 
    WHEN LOWER(`operator_type_id`) = 'private' THEN TRUE
    ELSE FALSE
  END AS `is_private`,
  CASE 
    WHEN `affiliation_type_ids` LIKE '%academic%' THEN TRUE
    WHEN LOWER(`organization_type`) LIKE '%university%' THEN TRUE
    WHEN LOWER(`organization_type`) LIKE '%college%' THEN TRUE
    WHEN LOWER(`name`) LIKE '%medical college%' THEN TRUE
    WHEN LOWER(`name`) LIKE '%institute of medical%' THEN TRUE
    ELSE FALSE
  END AS `is_academic`,
  CASE 
    WHEN `description` IS NOT NULL AND LENGTH(TRIM(`description`)) > 0 
    THEN SUBSTRING(TRIM(`description`), 1, 500)
    ELSE NULL
  END AS `description_summary`,
  CASE
    WHEN `recency_of_page_update` IS NOT NULL 
      AND DATEDIFF(CURRENT_DATE(), `recency_of_page_update`) <= 365 THEN 'Active'
    WHEN `recency_of_page_update` IS NOT NULL 
      AND DATEDIFF(CURRENT_DATE(), `recency_of_page_update`) > 1095 THEN 'Needs Review'
    WHEN `cluster_id` IS NULL THEN 'Needs Review'
    ELSE 'Active'
  END AS `status`,
  (
    CASE WHEN `unique_id` IS NOT NULL THEN 0.2 ELSE 0.0 END +
    CASE WHEN `facility_type_id` IS NOT NULL 
      AND `facility_type_id` NOT LIKE '%http%' 
      AND `facility_type_id` NOT LIKE '08f%' THEN 0.2 ELSE 0.0 END +
    CASE WHEN `operator_type_id` IS NOT NULL 
      AND `operator_type_id` NOT LIKE '%http%' 
      AND `operator_type_id` NOT LIKE '08f%' THEN 0.15 ELSE 0.0 END +
    CASE WHEN `address_city` IS NOT NULL THEN 0.1 ELSE 0.0 END +
    CASE WHEN `latitude` IS NOT NULL AND `longitude` IS NOT NULL THEN 0.15 ELSE 0.0 END +
    CASE WHEN `official_phone` IS NOT NULL OR `phone_numbers` IS NOT NULL THEN 0.1 ELSE 0.0 END +
    CASE WHEN `official_website` IS NOT NULL OR `websites` IS NOT NULL THEN 0.1 ELSE 0.0 END
  ) AS `gold_confidence_score`,
  COALESCE(
    `most_recent_social_media_post_date`,
    `recency_of_page_update`,
    `silver_ingested_at`
  ) AS `last_verified_at`,
  ARRAY_COMPACT(ARRAY(
    CASE WHEN `cluster_id` IS NULL THEN 'missing_cluster_id' ELSE NULL END,
    CASE WHEN `address_city` IS NULL THEN 'missing_address' ELSE NULL END,
    CASE WHEN `facility_type_id` IS NULL THEN 'missing_facility_type' ELSE NULL END,
    CASE WHEN `facility_type_id` LIKE '%http%' 
      OR `facility_type_id` LIKE '08f%' 
      OR `facility_type_id` RLIKE '^[0-9.]+$' THEN 'invalid_facility_type' ELSE NULL END,
    CASE WHEN `operator_type_id` LIKE '%http%' 
      OR `operator_type_id` LIKE '08f%' 
      OR `operator_type_id` RLIKE '^[0-9.]+$' THEN 'invalid_operator_type' ELSE NULL END,
    CASE WHEN `latitude` IS NULL OR `longitude` IS NULL THEN 'missing_coordinates' ELSE NULL END,
    CASE WHEN `source_types` LIKE '%dynamic%' THEN 'dynamic_source' ELSE NULL END,
    CASE WHEN `recency_of_page_update` IS NULL 
      OR DATEDIFF(CURRENT_DATE(), `recency_of_page_update`) > 730 THEN 'stale_data' ELSE NULL END
  )) AS `data_quality_flags`,
  CURRENT_TIMESTAMP() AS `created_at`,
  CURRENT_TIMESTAMP() AS `updated_at`
FROM `medallion_architecture`.`silver`.`facilities`;

num_affected_rows,num_inserted_rows
10077,10077


  File <command-4599543376089704>, line 2
    COUNT(*) as total_records,
    ^
IndentationError: unexpected indent
